# Finance Notebook: Credit Risk Stress Testing (Expanded)

This expanded notebook builds a synthetic retail credit portfolio and moves from predictive modeling to decision-grade risk analytics.

What is included:
- richer portfolio simulation with macro sensitivity,
- exploratory risk diagnostics,
- model benchmarking (logistic vs gradient boosting),
- calibration and stability checks,
- cost-aware threshold optimization,
- baseline and multi-scenario stress testing,
- segment-level expected-loss attribution and concentration analysis.

## 0) Imports and global settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(42)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1) Build a synthetic lending portfolio

In [ ]:
n = 12000

portfolio = pd.DataFrame(
    {
        "age": np.random.randint(21, 72, size=n),
        "income_k": np.random.lognormal(mean=3.75, sigma=0.52, size=n),
        "loan_amount_k": np.random.lognormal(mean=2.95, sigma=0.62, size=n),
        "dti": np.clip(np.random.normal(0.36, 0.13, size=n), 0.05, 0.95),
        "delinq_12m": np.random.poisson(0.45, size=n),
        "region": np.random.choice(["North", "South", "East", "West"], p=[0.23, 0.30, 0.24, 0.23], size=n),
        "employment_type": np.random.choice(["salaried", "self_employed", "contractor"], p=[0.65, 0.22, 0.13], size=n),
        "term_months": np.random.choice([24, 36, 48, 60, 72], p=[0.12, 0.26, 0.28, 0.24, 0.10], size=n),
    }
)

# Macro proxies that impact baseline risk
portfolio["macro_unemp"] = np.random.normal(5.2, 0.7, size=n)
portfolio["macro_rate"] = np.random.normal(3.1, 0.5, size=n)

income = portfolio["income_k"].to_numpy()
loan = portfolio["loan_amount_k"].to_numpy()

latent = (
    1.45 * portfolio["dti"].to_numpy()
    + 0.38 * np.log1p(portfolio["delinq_12m"].to_numpy())
    + 0.34 * (loan / (income + 1e-6))
    + 0.05 * (portfolio["term_months"].to_numpy() / 12)
    + 0.10 * portfolio["macro_unemp"].to_numpy()
    + 0.06 * portfolio["macro_rate"].to_numpy()
    + np.where(portfolio["region"].to_numpy() == "South", 0.20, 0.0)
    + np.where(portfolio["employment_type"].to_numpy() == "contractor", 0.16, 0.0)
    - np.where(portfolio["employment_type"].to_numpy() == "salaried", 0.08, 0.0)
)

pd_true = 1 / (1 + np.exp(-(latent - 1.8)))
portfolio["default"] = np.random.binomial(1, pd_true, size=n)

# EAD and LGD assumptions
rng = np.random.default_rng(42)
portfolio["ead_k"] = portfolio["loan_amount_k"] * rng.uniform(0.90, 1.25, size=n)
portfolio["lgd"] = np.clip(rng.normal(0.46, 0.12, size=n), 0.10, 0.90)

portfolio.head()

In [ ]:
print(f"Portfolio size: {len(portfolio):,}")
print(f"Observed default rate: {portfolio['default'].mean():.2%}")
print(f"Average EAD (k): {portfolio['ead_k'].mean():.2f}")

## 2) Exploratory diagnostics

In [ ]:
risk_profile = (
    portfolio.groupby("region", as_index=False)
    .agg(
        obs_default_rate=("default", "mean"),
        avg_dti=("dti", "mean"),
        avg_delinq=("delinq_12m", "mean"),
        exposure_k=("ead_k", "sum"),
    )
    .sort_values("obs_default_rate", ascending=False)
)
risk_profile

In [ ]:
portfolio["dti_bin"] = pd.qcut(portfolio["dti"], q=10, duplicates="drop")
default_by_dti = portfolio.groupby("dti_bin", observed=False)["default"].mean().reset_index()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=risk_profile, x="region", y="obs_default_rate", ax=ax[0], palette="Blues_d")
ax[0].set_title("Observed Default Rate by Region")
ax[0].set_ylabel("Default Rate")

sns.lineplot(data=default_by_dti, x=default_by_dti.index, y="default", marker="o", ax=ax[1], color="#C44E52")
ax[1].set_title("Default Rate Across DTI Deciles")
ax[1].set_xlabel("DTI decile")
ax[1].set_ylabel("Default Rate")

plt.tight_layout()
plt.show()

## 3) Train-test split and preprocessing

In [ ]:
features = [
    "age",
    "income_k",
    "loan_amount_k",
    "dti",
    "delinq_12m",
    "region",
    "employment_type",
    "term_months",
    "macro_unemp",
    "macro_rate",
]

target = "default"

X = portfolio[features]
y = portfolio[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

num_cols = [
    "age",
    "income_k",
    "loan_amount_k",
    "dti",
    "delinq_12m",
    "term_months",
    "macro_unemp",
    "macro_rate",
]
cat_cols = ["region", "employment_type"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

## 4) Benchmark models (Logistic vs HistGradientBoosting)

In [ ]:
logit_model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

hgb_model = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "clf",
            HistGradientBoostingClassifier(
                learning_rate=0.06,
                max_depth=4,
                min_samples_leaf=40,
                max_iter=250,
                random_state=42,
            ),
        ),
    ]
)

logit_model.fit(X_train, y_train)
hgb_model.fit(X_train, y_train)

preds = {
    "Logistic": logit_model.predict_proba(X_test)[:, 1],
    "HistGB": hgb_model.predict_proba(X_test)[:, 1],
}

metrics = []
for name, p in preds.items():
    metrics.append(
        {
            "model": name,
            "roc_auc": roc_auc_score(y_test, p),
            "pr_auc": average_precision_score(y_test, p),
            "brier": brier_score_loss(y_test, p),
        }
    )

metrics_df = pd.DataFrame(metrics).sort_values("roc_auc", ascending=False)
metrics_df

In [ ]:
plt.figure(figsize=(7, 4))
plot_df = metrics_df.melt(id_vars="model", value_vars=["roc_auc", "pr_auc", "brier"], var_name="metric")
sns.barplot(data=plot_df, x="metric", y="value", hue="model", palette="Set2")
plt.title("Model Quality Comparison")
plt.tight_layout()
plt.show()

## 5) Pick champion model and assess calibration

In [ ]:
champion_name = metrics_df.iloc[0]["model"]
champion = logit_model if champion_name == "Logistic" else hgb_model
proba_test = preds[champion_name]

print(f"Champion model: {champion_name}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, proba_test):.3f}")
print(f"Test PR-AUC : {average_precision_score(y_test, proba_test):.3f}")
print(f"Brier score : {brier_score_loss(y_test, proba_test):.4f}")

frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=12)
plt.figure(figsize=(5.8, 5.4))
plt.plot(mean_pred, frac_pos, marker="o", label="Champion")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted PD")
plt.ylabel("Observed default rate")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.show()

## 6) Cost-aware threshold optimization

In [ ]:
# Assume binary decision: review/high-touch vs standard underwriting
# Cost setup in monetary units (relative scale)
cost_fp = 1.0   # unnecessary review effort
cost_fn = 12.0  # missed default impact proxy

threshold_grid = np.linspace(0.05, 0.80, 120)
rows = []
for t in threshold_grid:
    pred = (proba_test >= t).astype(int)
    tp = int(((pred == 1) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    fn = int(((pred == 0) & (y_test == 1)).sum())
    tn = int(((pred == 0) & (y_test == 0)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    total_cost = cost_fp * fp + cost_fn * fn

    rows.append(
        {
            "threshold": t,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": precision,
            "recall": recall,
            "total_cost": total_cost,
        }
    )

th_table = pd.DataFrame(rows)
best = th_table.sort_values("total_cost", ascending=True).iloc[0]

print(f"Optimal threshold by cost: {best['threshold']:.3f}")
print(f"Total cost at optimum: {best['total_cost']:.1f}")
print(f"Precision={best['precision']:.3f}, Recall={best['recall']:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].plot(th_table["threshold"], th_table["total_cost"], color="#C44E52")
ax[0].axvline(best["threshold"], linestyle="--", color="black")
ax[0].set_title("Threshold vs Total Cost")
ax[0].set_xlabel("Threshold")
ax[0].set_ylabel("Total cost")

ax[1].plot(th_table["threshold"], th_table["precision"], label="Precision", color="#4C72B0")
ax[1].plot(th_table["threshold"], th_table["recall"], label="Recall", color="#55A868")
ax[1].axvline(best["threshold"], linestyle="--", color="black")
ax[1].set_title("Precision/Recall Tradeoff")
ax[1].set_xlabel("Threshold")
ax[1].legend()

plt.tight_layout()
plt.show()

## 7) Baseline expected loss and ranking diagnostics

In [ ]:
portfolio_scored = portfolio.copy()
portfolio_scored["pd_hat"] = champion.predict_proba(portfolio_scored[features])[:, 1]
portfolio_scored["el_k"] = portfolio_scored["pd_hat"] * portfolio_scored["lgd"] * portfolio_scored["ead_k"]

baseline_summary = pd.DataFrame(
    {
        "metric": [
            "mean_pd",
            "total_ead_k",
            "total_expected_loss_k",
            "avg_expected_loss_per_loan_k",
        ],
        "value": [
            portfolio_scored["pd_hat"].mean(),
            portfolio_scored["ead_k"].sum(),
            portfolio_scored["el_k"].sum(),
            portfolio_scored["el_k"].mean(),
        ],
    }
)
baseline_summary

In [ ]:
# Concentration: top decile of predicted PD
portfolio_scored["pd_decile"] = pd.qcut(portfolio_scored["pd_hat"], q=10, labels=False, duplicates="drop")
concentration = (
    portfolio_scored.groupby("pd_decile", as_index=False)
    .agg(exposure_k=("ead_k", "sum"), expected_loss_k=("el_k", "sum"))
    .sort_values("pd_decile")
)
concentration["loss_share"] = concentration["expected_loss_k"] / concentration["expected_loss_k"].sum()
concentration

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(concentration["pd_decile"].astype(str), concentration["loss_share"], color="#8172B2")
plt.xlabel("Predicted PD decile (0=lowest risk)")
plt.ylabel("Share of expected loss")
plt.title("Expected Loss Concentration by PD Decile")
plt.tight_layout()
plt.show()

## 8) Multi-scenario stress testing

In [ ]:
def apply_stress(df: pd.DataFrame, scenario: str) -> pd.DataFrame:
    out = df.copy()

    if scenario == "baseline":
        return out
    if scenario == "mild":
        out["macro_unemp"] = out["macro_unemp"] + 0.8
        out["macro_rate"] = out["macro_rate"] + 0.5
        out["dti"] = np.clip(out["dti"] * 1.08, 0.05, 0.99)
        out["delinq_12m"] = np.minimum(out["delinq_12m"] + np.random.binomial(1, 0.20, size=len(out)), 12)
        return out
    if scenario == "severe":
        out["macro_unemp"] = out["macro_unemp"] + 1.8
        out["macro_rate"] = out["macro_rate"] + 1.2
        out["dti"] = np.clip(out["dti"] * 1.20, 0.05, 0.99)
        out["delinq_12m"] = np.minimum(out["delinq_12m"] + np.random.binomial(1, 0.40, size=len(out)), 12)
        out["income_k"] = np.clip(out["income_k"] * 0.94, 0.01, None)
        return out
    raise ValueError(f"Unknown scenario: {scenario}")

scenario_rows = []
for scn in ["baseline", "mild", "severe"]:
    df_s = apply_stress(portfolio_scored[features + ["ead_k", "lgd"]], scn)
    pd_s = champion.predict_proba(df_s[features])[:, 1]
    el_s = pd_s * df_s["lgd"].to_numpy() * df_s["ead_k"].to_numpy()

    scenario_rows.append(
        {
            "scenario": scn,
            "avg_pd": pd_s.mean(),
            "total_expected_loss_k": el_s.sum(),
            "p95_pd": np.quantile(pd_s, 0.95),
        }
    )

scenario_summary = pd.DataFrame(scenario_rows)
base_loss = scenario_summary.loc[scenario_summary["scenario"] == "baseline", "total_expected_loss_k"].iloc[0]
scenario_summary["loss_uplift_pct_vs_base"] = (scenario_summary["total_expected_loss_k"] / base_loss - 1) * 100
scenario_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=scenario_summary, x="scenario", y="total_expected_loss_k", ax=ax[0], palette="Reds")
ax[0].set_title("Expected Loss by Scenario")
ax[0].set_ylabel("Expected loss (k)")

sns.lineplot(data=scenario_summary, x="scenario", y="avg_pd", marker="o", ax=ax[1], color="#4C72B0")
ax[1].set_title("Average PD by Scenario")
ax[1].set_ylabel("Average PD")

plt.tight_layout()
plt.show()

## 9) Stress attribution by region and employment type

In [ ]:
baseline_df = apply_stress(portfolio_scored.copy(), "baseline")
severe_df = apply_stress(portfolio_scored.copy(), "severe")

baseline_df["pd"] = champion.predict_proba(baseline_df[features])[:, 1]
severe_df["pd"] = champion.predict_proba(severe_df[features])[:, 1]

baseline_df["el"] = baseline_df["pd"] * baseline_df["lgd"] * baseline_df["ead_k"]
severe_df["el"] = severe_df["pd"] * severe_df["lgd"] * severe_df["ead_k"]

seg_base = baseline_df.groupby(["region", "employment_type"], as_index=False)["el"].sum().rename(columns={"el": "el_baseline"})
seg_sev = severe_df.groupby(["region", "employment_type"], as_index=False)["el"].sum().rename(columns={"el": "el_severe"})
seg = seg_base.merge(seg_sev, on=["region", "employment_type"])
seg["uplift_pct"] = (seg["el_severe"] / seg["el_baseline"] - 1) * 100
seg.sort_values("uplift_pct", ascending=False).head(12)

In [ ]:
plt.figure(figsize=(9, 4.5))
plot_seg = seg.sort_values("uplift_pct", ascending=False).head(10).copy()
plot_seg["segment"] = plot_seg["region"] + " | " + plot_seg["employment_type"]

sns.barplot(data=plot_seg, x="uplift_pct", y="segment", palette="viridis")
plt.xlabel("Severe stress EL uplift (%)")
plt.ylabel("Segment")
plt.title("Top Segments by Stress Sensitivity")
plt.tight_layout()
plt.show()

## 10) Final summary

- The expanded pipeline now links borrower-level PD modeling to portfolio-level stress economics.
- Model selection is based on discrimination and calibration, not ROC-AUC alone.
- Threshold selection is tied to business cost, improving operational relevance.
- Severe stress testing and segment attribution identify where underwriting and monitoring should be tightened first.